# Train the poem model on Colab (free GPU)

Same code as the laptop project (`src/`), just on an NVIDIA GPU so the Mac stays cool.

**Steps:** Runtime → Change runtime type → **T4 GPU** → Run all.
You will upload `colab_bundle.zip` (images + code) and at the end download `colab_results.zip`.
Then on the laptop: unzip it into the project folder (see README).

## 1. Check the GPU

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Upload `colab_bundle.zip` (from the project folder)

In [ ]:
import os, zipfile
from google.colab import files
os.makedirs("/content/project", exist_ok=True)
%cd /content/project
up = files.upload()                      # pick colab_bundle.zip
name = next(iter(up))
zipfile.ZipFile(name).extractall("/content/project")
!ls /content/project /content/project/data
!for m in happy sad calm energetic; do echo $m $(ls data/$m | wc -l); done

## 3. Install the same library versions as the laptop

In [ ]:
!pip install -q transformers==4.46.3 peft==0.13.2 accelerate==1.1.1 pronouncing==0.2.0 "setuptools<81"

## 4. Build the image → poem dataset
Caption every image, write 8 candidate poems with the strict teacher prompt, score them, keep the best.
Split by **image** 80/10/10 (test images get no poems). ~10 min on a T4.

In [ ]:
%cd /content/project
!python -u -m src.poem_data

## 5. Fine-tune the poem model with LoRA (supervised)
x = short prompt (caption + mood), y = poem. Loss on poem tokens only. ~3–5 min on a T4.

In [ ]:
!python -u -m src.train_poet

## 6. Evaluate: base vs fine-tuned on unseen test images

In [ ]:
!python -u -m src.eval_poet

In [ ]:
from IPython.display import Image, display
for f in ["poet_loss", "poet_before_after", "poet_line_counts", "poet_clip"]:
    display(Image(f"reports/{f}.png"))

## 7. Download the results → unzip into the project folder on the laptop

In [ ]:
import shutil, os
os.makedirs("/content/out", exist_ok=True)
!cd /content/project && zip -qr /content/colab_results.zip data/poems models/poet_lora reports/poet_loss.csv reports/poet_eval.csv reports/poet_eval_summary.csv reports/poet_*.png
!unzip -l /content/colab_results.zip | tail -15
files.download("/content/colab_results.zip")